# 02 Equivalent Image Generation

This notebook generates an **equivalent image** — an image that produces the same classifier output as the original, but with its null-space removed.

Pipeline:
1. Extract classifier features from an input image
2. Decompose the classifier head with SVD to get null/principal subspaces
3. Project features onto the principal subspace (remove null-space)
4. Translate both original and principal features into the Karlo embedding space
5. Generate images from both embeddings using UnCLIP (Kakao Karlo)
6. Compare: original reconstruction vs. equivalent (null-space-free) image

**Part A** — step-by-step manual pipeline (explicit, educational)  
**Part B** — same result via `SingleImageAnalyzer` (high-level API)

In [ ]:
#---------------------------- setup install colab --------------------------------#
!git clone https://github.com/harel314/SING-analyzing-semantic-invariants-classifiers.git
%cd SING-analyzing-semantic-invariants-classifiers
%pip install .

## LOCAL USERS
simply run `uv sync` and continue from here

In [ ]:
#---------------------------- gpu check --------------------------------#
!nvidia-smi

In [ ]:
#---------------------------- imports and config --------------------------------#
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

from sing.core.projectors import compute_projectors_from_weight, principal_component
from sing.generation.generate import generate_seed_set
from sing.generation.unclip_wrapper import KakaoUnclipWrapper
from sing.models.registry import load_model
from sing.translators.registry import load_default_translator

try:
    import google.colab
    repo_root = Path(".").resolve()
except ImportError:
    repo_root = Path("..").resolve()

image_path = repo_root / "samples" / "border_collie_n02106166.jpeg"
if not image_path.exists():
    raise FileNotFoundError(f"Set image_path to an existing image. Missing: {image_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")

---
## Part A — Step-by-step manual pipeline

In [ ]:
#---------------------------- load model and translator --------------------------------#
model_name = "resnet"

loaded_model = load_model(model_name=model_name, device=device)
loaded_translator = load_default_translator(
    translators_root=repo_root / "translators",
    registry_path=repo_root / "translators" / "registry.yaml",
    model_name=loaded_model.name,
    device=device,
)
print(f"model={loaded_model.name}")
print(f"translator={loaded_translator.metadata.translator_name} architecture={loaded_translator.metadata.architecture}")

In [ ]:
#---------------------------- extract features and project --------------------------------#
image = Image.open(image_path).convert("RGB")
input_tensor = loaded_model.preprocess(image).unsqueeze(0).to(device)

with torch.no_grad():
    features = loaded_model.wrapper.extract_features(input_tensor)
    classifier_weight = loaded_model.wrapper.classifier_weight.detach().to(device)
    projectors = compute_projectors_from_weight(classifier_weight)
    principal_features = principal_component(features, projectors.v_null)

print(f"features={tuple(features.shape)}")
print(f"projector_rank={projectors.rank} v_null={tuple(projectors.v_null.shape)} v_principal={tuple(projectors.v_principal.shape)}")
print(f"principal_features={tuple(principal_features.shape)}")

In [ ]:
#---------------------------- translate to embedding space --------------------------------#
with torch.no_grad():
    translated_original = loaded_translator.model(features)
    translated_principal = loaded_translator.model(principal_features)

print(f"translated_original={tuple(translated_original.shape)}")
print(f"translated_principal={tuple(translated_principal.shape)}")

In [ ]:
#---------------------------- load unclip and generate --------------------------------#
seeds = [42, 1337]
output_dir = repo_root / "outputs" / "notebook_generation_manual"

unclip = KakaoUnclipWrapper(
    device=device,
    torch_dtype=KakaoUnclipWrapper.default_dtype(device),
)

results = generate_seed_set(
    wrapper=unclip,
    image=image,
    principal_embedding=translated_principal.detach(),
    seeds=seeds,
    output_dir=output_dir,
)

print(f"generated {len(results)} seed pair(s) -> {output_dir}")

In [ ]:
#---------------------------- display results --------------------------------#
fig, axes = plt.subplots(len(results) + 1, 2, figsize=(8, 4 * (len(results) + 1)))

axes[0, 0].imshow(image)
axes[0, 0].set_title("input image")
axes[0, 0].axis("off")
axes[0, 1].axis("off")

for row, result in enumerate(results, start=1):
    axes[row, 0].imshow(Image.open(result.original_path))
    axes[row, 0].set_title(f"seed={result.seed} | original reconstruction")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(Image.open(result.principal_path))
    axes[row, 1].set_title(f"seed={result.seed} | equivalent (null-space removed)")
    axes[row, 1].axis("off")

plot_path = repo_root / "outputs" / "notebook_generation_manual.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"saved plot: {plot_path}")
plt.show()

---
## Part B — High-level API via `SingleImageAnalyzer`

The `SingleImageAnalyzer` runs the exact same pipeline in one call — feature extraction, SVD, translation, and generation — and additionally computes IS/AS scores and dictionary similarity.

## If you previously ran Part A

In [ ]:
#---------------------------- cleanup before part B --------------------------------#
del unclip
import gc
gc.collect()
torch.cuda.empty_cache()
print("VRAM freed")


## If you did not initiate Part A, you may start from here

In [ ]:
#---------------------------- run via SingleImageAnalyzer --------------------------------#
from sing.core.analyzer import SingleImageAnalyzer

analyzer = SingleImageAnalyzer(repo_root=repo_root, device=str(device))

output = analyzer.analyze(
    image_path=image_path,
    model_name="resnet",
    seeds=[42, 1337],
    output_dir=repo_root / "outputs" / "notebook_generation_highlevel",
)

print(f"model={output.model_name}  translator={output.translator_name}")
print(f"IS={output.is_value:.6f}")
if output.as_value is not None:
    print(f"AS={output.as_value:.6f}")
print(f"generated {len(output.generated_files)} seed pair(s)")
for r in output.generated_files:
    print(f"  seed={r.seed}  original={r.original_path.name}  principal={r.principal_path.name}")

In [ ]:
#---------------------------- display results --------------------------------#
fig, axes = plt.subplots(len(output.generated_files) + 1, 2, figsize=(8, 4 * (len(output.generated_files) + 1)))

axes[0, 0].imshow(image)
axes[0, 0].set_title("input image")
axes[0, 0].axis("off")
axes[0, 1].axis("off")

for row, result in enumerate(output.generated_files, start=1):
    axes[row, 0].imshow(Image.open(result.original_path))
    axes[row, 0].set_title(f"seed={result.seed} | original reconstruction")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(Image.open(result.principal_path))
    axes[row, 1].set_title(f"seed={result.seed} | equivalent (null-space removed)")
    axes[row, 1].axis("off")

plot_path = repo_root / "outputs" / "notebook_generation_highlevel.png"
plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"saved plot: {plot_path}")
plt.show()